In [1]:
import pandas as pd
import ast
import json
import time

In [2]:
df_main = pd.read_csv('../data/processed/oscar_tmdb_cleaned.csv')
credits_df = pd.read_csv('../data/raw/tmdb_5000_credits.csv')

In [3]:
credits_df.columns

Index(['movie_id', 'title', 'cast', 'crew'], dtype='str')

In [4]:
crew_keys = json.loads(credits_df['crew'][0])

crew_keys[0].keys()

dict_keys(['credit_id', 'department', 'gender', 'id', 'job', 'name'])

In [5]:
cast_keys = json.loads(credits_df['cast'][0])

cast_keys[0].keys()

dict_keys(['cast_id', 'character', 'credit_id', 'gender', 'id', 'name', 'order'])

In [6]:
def get_director(crew_data):
    try:
        crew_list = ast.literal_eval(crew_data)
        for person in crew_list:
            if person['job'] == 'Director':
                return person['name']
        return "Unknown"
    except (ValueError, SyntaxError, TypeError):
        return "Unknown"

In [7]:
def get_top_actors(cast_data):
    try:
        cast_list = ast.literal_eval(cast_data)
        return [person['name'] for person in cast_list[:3]]
    except (ValueError, SyntaxError, TypeError):
        return []

In [8]:
credits_df['director'] = credits_df['crew'].apply(get_director)
credits_df['lead_actors'] = credits_df['cast'].apply(get_top_actors)

In [9]:
df_merged = pd.merge(df_main, credits_df[['movie_id', 'director', 'lead_actors']], left_on='id', right_on='movie_id',
                     how='inner')

In [10]:
director_stats = df_merged.groupby('director')['is_oscar_winner'].mean().reset_index()

director_stats.rename(columns={'is_oscar_winner': 'director_oscar_rate'}, inplace=True)

In [11]:
director_stats.head(5)

,director,director_oscar_rate
0,Adam McKay,1.000000
1,Adrian Lyne,0.333333
2,Alan Parker,0.500000
3,Alejandro Amenábar,1.000000
4,Alejandro González Iñárritu,0.400000


In [12]:
actors_expanded = credits_df[['movie_id', 'lead_actors']].explode('lead_actors')

actors_expanded.rename(columns={'lead_actors': 'actor_name'}, inplace=True)

In [13]:
actors_expanded.head(3)

,movie_id,actor_name
0,19995,Sam Worthington
0,19995,Zoe Saldana
0,19995,Sigourney Weaver


In [14]:
actors_with_oscar = pd.merge(actors_expanded, df_main[['id', 'is_oscar_winner']], left_on='movie_id', right_on='id',
                             how='inner')

In [15]:
actor_stats = actors_with_oscar.groupby('actor_name')['is_oscar_winner'].mean().reset_index()

actor_stats.rename(columns={'is_oscar_winner': 'actor_oscar_rate'}, inplace=True)

In [16]:
actors_mapped = pd.merge(actors_expanded, actor_stats, on='actor_name', how='left')

In [17]:
movie_actor_rate = actors_mapped.groupby('movie_id')['actor_oscar_rate'].mean().reset_index()

movie_actor_rate.rename(columns={'actor_oscar_rate': 'avg_cast_oscar_rate'}, inplace=True)

In [18]:
df_final = pd.merge(df_merged, director_stats, on='director', how='left')

df_final = pd.merge(df_final, movie_actor_rate, on='movie_id', how='left')

cols_to_drop = ['director', 'lead_actors', 'movie_id', 'id']
df_final.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [19]:
df_final['director_oscar_rate'] = df_final['director_oscar_rate'].fillna(0)
df_final['avg_cast_oscar_rate'] = df_final['avg_cast_oscar_rate'].fillna(0)

In [20]:
df_final.to_csv('../data/processed/oscar_tmdb_advanced_features.csv', index=False)

In [21]:
df_final.columns[-5:]

Index(['key_gilbert_and_sullivan', 'key_nightgown', 'key_nosferatu',
       'director_oscar_rate', 'avg_cast_oscar_rate'],
      dtype='str')